<a href="https://colab.research.google.com/github/JaeMoonJeong/AI-ML-Portfolio/blob/Jae-Moon/11_3%ED%8C%80_%EC%A0%95%EC%9E%AC%EB%AC%B8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🚀 Mission 11: 기계 번역 (Machine Translation)

## 0. 미션 설명

**[개요]**
이번 미션은 **기계 번역(Machine Translation) 실습**으로, 한국어 문장을 영어로 번역하는 모델을 구축하는 프로젝트입니다. 총 3가지 모델(**Seq2Seq 기본**, **Attention 적용**, **성능 비교**)을 구현하고 학습시키며, 각 모델의 성능을 분석하는 것이 핵심 목표입니다.

### 1. 사용 데이터셋
* **데이터 형식:** JSON 파일 (Key: `"ko"`: 한국어, `"mt"`: 영어)
* **다운로드:** 미션 11 데이터 셋 활용
* **파일 경로 예시:**
    * 훈련 데이터: `.../일상생활및구어체_한영_train_set.json`
    * 검증 데이터: `.../일상생활및구어체_한영_valid_set.json`
    *(※ 파일 경로는 본인의 환경에 맞게 수정 필요)*

### 2. 가이드라인

#### 1) 데이터 전처리
* **토크나이저(Tokenizer):** 한국어 및 영어 문장에 적합한 토크나이저를 선택하여 토큰화 진행
* **길이 설정:** 문장 길이를 분석하여 최대 길이(`MAX_LENGTH`) 설정
* **특수 토큰:** 필요시 `SOS`(Start), `EOS`(End), `PAD`(Padding), `UNK`(Unknown) 정의

#### 2) 어휘 사전 구축
* **사전 구성:** 한국어/영어 각각의 어휘 사전 생성
* **빈도 고려:** 단어 등장 빈도를 고려하여 임베딩 및 모델 구성에 활용

#### 3) 텐서 변환 및 데이터 로더
* **인덱싱 & 패딩:** 문장을 인덱스 시퀀스로 변환 후 `PAD` 토큰으로 고정 길이 패딩
* **DataLoader:** `TensorDataset`과 `DataLoader`를 활용하여 배치(Batch) 단위 처리 구현

#### 4) 모델 구현 및 학습
* **Seq2Seq 모델 (Baseline):**
    * GRU 기반 Encoder-Decoder 구조
    * **Teacher Forcing** 기법 적용
* **Attention 모델:**
    * Bahdanau 또는 Luong Attention을 적용한 디코더 구현
    * 기본 모델 대비 번역 성능 향상 검증

#### 5) 모델 학습 및 평가
* **학습 진행:** 각 모델별 학습 수행
* **평가 함수:** 무작위 문장 쌍에 대한 번역 결과 출력 및 정성적 확인
* **결과 정제:** 출력 문장에서 특수 토큰 등을 제거하여 최종 문장 도출

### 3. 추가 실험 (선택 사항)
* **전처리 개선:** 불용어(Stopwords) 제거, 텍스트 정규화 등
* **모델 구조 변경:** 레이어 수, 은닉 상태(Hidden State) 크기 조절, Attention 기법 수정
* **하이퍼파라미터 튜닝:** 학습률(Learning Rate), 배치 크기(Batch Size) 등 최적화
* **정량적 평가:** BLEU Score 등 평가지표 도입

### 4. 제출 안내
* **파일명:** `11_{팀명}_{성함}.ipynb`
* **필수 포함 내용:**
    1.  **모델 구현 및 학습 결과:** (로드 → 전처리 → 임베딩 → 모델링 → 평가) 전 과정
    2.  **Markdown 설명:** 각 코드 셀의 의도, 알고리즘, 함수 설명 상세 기록
    3.  **성능 평가:** 테스트 데이터에 대한 번역 결과 및 정성적/정량적 분석

### 5. 참고 사항
* **Baseline Code:** 초기 모델 구성을 돕기 위한 기본 코드 제공 (링크 참고)
* **주의:** Baseline은 참고용이며, 이를 그대로 제출하기보다 본인의 아이디어를 더해 발전시키는 것이 중요합니다.

## 1. 기본 환경세팅

### (1) 환경 및 데이터 파일 점검

In [2]:
import torch
import os

print("=== 1. 주방 도구(Mac GPU) 점검 ===")

# Mac의 MPS(Metal Performance Shaders) 가속이 가능한지 확인
if torch.backends.mps.is_available():
    device = torch.device("mps")
    print(f"✅ MPS를 사용합니다! 🚀")
else:
    device = torch.device("cpu")
    print("⚠️ MPS를 사용할 수 없어 휴대용 CPU를 사용합니다.")

print(f"   -> 결정된 장치: {device}")

=== 1. 주방 도구(Mac GPU) 점검 ===
⚠️ MPS를 사용할 수 없어 휴대용 CPU를 사용합니다.
   -> 결정된 장치: cpu


### (2) 데이터 박스 열어보기 (Load & Inspect)

In [3]:
import json
import random
# pandas는 현재 코드에 필요 없으므로 주석 처리하거나 삭제합니다.
# import pandas as pd

# 1. Google Drive 마운트 (코랩에서 필수)
# 이 코드를 실행하면 인증을 요구하며, 드라이브 파일에 접근 가능해집니다.
from google.colab import drive
print("🔗 Google Drive 마운트 중...")
drive.mount('/content/drive')
print("✅ 마운트 완료!")

#상대경로 / 절대경로
#df = pd.read_csv('./content/drive/MyDrive/00. MIT/07. Artifical Intelligence/스프린트 미션 11/일상생활및구어체_한영_train_set.json')


🔗 Google Drive 마운트 중...
Mounted at /content/drive
✅ 마운트 완료!


In [4]:
# 2. 파일 경로 변수 설정 (사용자의 실제 경로에 맞게 수정 필요)
# 경로를 사용자가 지정한 것처럼 드라이브 내 특정 폴더로 지정합니다.
# **주의: '00. MIT/07. Artifical Intelligence/스프린트 미션 11/' 이 경로가 정확한지 확인해 주세요.**
base_path = '/content/drive/MyDrive/00. MIT/07. Artifical Intelligence/스프린트 미션 11/'
train_filename = base_path + '일상생활및구어체_한영_train_set.json'
valid_filename = base_path + '일상생활및구어체_한영_valid_set.json' # valid 파일명도 가정하여 추가했습니다.

# 3. 데이터 로드 함수
def load_json_data(file_path):
    """
    지정된 경로의 JSON 파일을 로드하고, 내부 'data' 키의 리스트를 반환합니다.
    """
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
        # 데이터셋 구조: {'data': [ ... 실제 리스트 ... ]}
        if 'data' in data and isinstance(data['data'], list):
            return data['data']
        else:
            print(f"⚠️ 경고: {file_path} 파일의 최상위 키가 'data'가 아니거나 리스트 형태가 아닙니다.")
            return []
    except FileNotFoundError:
        print(f"❌ 오류: 파일을 찾을 수 없습니다. 경로를 확인하세요: {file_path}")
        return []
    except json.JSONDecodeError:
        print(f"❌ 오류: JSON 디코딩에 실패했습니다. 파일 내용이 올바른 JSON 형식이 아닙니다: {file_path}")
        return []


In [5]:
# 4. 파일 읽어오기 및 확인
print("\n📦 데이터를 박스에서 꺼내는 중...")
# 파일이 없거나 오류 발생 시 빈 리스트가 반환되므로 오류 처리 함수를 사용합니다.
train_data = load_json_data(train_filename)
valid_data = load_json_data(valid_filename)


📦 데이터를 박스에서 꺼내는 중...


In [6]:
# 5. 랜덤으로 하나 찍어서 맛보기 (제대로 짝이 맞는지 확인)

if train_data and valid_data:
    print(f"✅ 로드 완료!")
    print(f"   - 훈련 데이터(Train): {len(train_data)} 문장")
    print(f"   - 검증 데이터(Valid): {len(valid_data)} 문장")


    print("\n🔍 [랜덤 샘플 확인]")
    sample = random.choice(train_data)
    print(f"🇰🇷 한국어: **{sample['ko']}**")
    print(f"🇺🇸 영  어: **{sample['mt']}**")
else:
    print("🚨 데이터 로드에 문제가 발생하여 샘플을 확인할 수 없습니다. 위의 오류 메시지를 확인해주세요.")

✅ 로드 완료!
   - 훈련 데이터(Train): 1200000 문장
   - 검증 데이터(Valid): 150000 문장

🔍 [랜덤 샘플 확인]
🇰🇷 한국어: **아 다른 일정이 있나봐요?**
🇺🇸 영  어: **Oh, you must have something else to do?**


### (3) EDA

A. 필수 라이브러리 및 도구 불러오기
- 이 미션은 한국어-영어 번역이므로, 각 언어에 특화된 토크나이저(Tokenizer)가 필요합니다. 한국어는 형태소 분석기인 Okt를, 영어는 nltk의 word_tokenize를 사용합니다.

In [7]:
!apt-get install -y mecab mecab-ipadic-utf8 libmecab-dev
!pip install konlpy

import torch
import torch.nn as nn
import torch.optim as optim
import random
import numpy as np
from torch.utils.data import DataLoader, TensorDataset, RandomSampler
from konlpy.tag import Okt
import nltk
from nltk.tokenize import word_tokenize
# ... (중략) ...
nltk.download('punkt')
# ...

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  libmecab2 mecab-ipadic mecab-utils
The following NEW packages will be installed:
  libmecab-dev libmecab2 mecab mecab-ipadic mecab-ipadic-utf8 mecab-utils
0 upgraded, 6 newly installed, 0 to remove and 41 not upgraded.
Need to get 7,367 kB of archives.
After this operation, 59.3 MB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 libmecab2 amd64 0.996-14build9 [199 kB]
Get:2 http://archive.ubuntu.com/ubuntu jammy/main amd64 libmecab-dev amd64 0.996-14build9 [306 kB]
Get:3 http://archive.ubuntu.com/ubuntu jammy/main amd64 mecab-utils amd64 0.996-14build9 [4,850 B]
Get:4 http://archive.ubuntu.com/ubuntu jammy/main amd64 mecab-ipadic all 2.7.0-20070801+main-3 [6,718 kB]
Get:5 http://archive.ubuntu.com/ubuntu jammy/universe amd64 mecab amd64 0.996-14build9 [136 kB]
Get:6 http://archive.ubuntu.co

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


True

In [8]:
# ko와 mt 데이터 추출 (5만 개 샘플만 사용)
MAX_SAMPLES = 50000
ko_sentences_train = [item["ko"] for item in train_data][:MAX_SAMPLES]
mt_sentences_train = [item["mt"] for item in train_data][:MAX_SAMPLES]
ko_sentences_valid = [item["ko"] for item in valid_data]
mt_sentences_valid = [item["mt"] for item in valid_data]

# 한국어 및 영어 토크나이저 함수 정의
tokenizer_ko = Okt().morphs
tokenizer_en = word_tokenize

In [9]:
import nltk
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [10]:
## 문장 길이 분석
ko_lengths = [len(tokenizer_ko(sent)) for sent in ko_sentences_train]
en_lengths = [len(tokenizer_en(sent)) for sent in mt_sentences_train]
all_lengths = ko_lengths + en_lengths

# 한국어와 영어 중 가장 긴 문장의 길이 기준으로 MAX_LENGTH 설정
MAX_LENGTH = max(max(ko_lengths), max(en_lengths)) + 1  # SOS, EOS 포함 고려
print(f"Max sequence length: {MAX_LENGTH}")

Max sequence length: 96


In [11]:
# 특수 토큰 정의
SOS_token = 0  # Start Of Sequence
EOS_token = 1  # End Of Sequence
PAD_token = 2  # Padding
UNK_token = 3  # Unknown

In [12]:
class Lang:
    def __init__(self, name):
        self.name = name
        # 초기에는 PAD, SOS, EOS, UNK 토큰을 미리 등록
        self.word2index = {PAD_token: "PAD", SOS_token: "SOS", EOS_token: "EOS", UNK_token: "UNK"}
        self.index2word = {0: "PAD", 1: "SOS", 2: "EOS", 3: "UNK"}
        self.word2count = {}
        self.n_words = 4  # PAD, SOS, EOS, UNK 포함

    def addSentence(self, sentence, tokenizer):
        for word in tokenizer(sentence):
            self.addWord(word)

    def addWord(self, word):
        if word not in self.word2index:
            self.word2index[word] = self.n_words
            self.index2word[self.n_words] = word
            self.word2count[word] = 1
            self.n_words += 1
        else:
            self.word2count[word] += 1

In [13]:
class Lang:
    def __init__(self, name):
        self.name = name
        # 초기에는 PAD, SOS, EOS, UNK 토큰을 미리 등록
        self.word2index = {PAD_token: "PAD", SOS_token: "SOS", EOS_token: "EOS", UNK_token: "UNK"}
        self.index2word = {0: "PAD", 1: "SOS", 2: "EOS", 3: "UNK"}
        self.word2count = {}
        self.n_words = 4  # PAD, SOS, EOS, UNK 포함

    def addSentence(self, sentence, tokenizer):
        for word in tokenizer(sentence):
            self.addWord(word)

    def addWord(self, word):
        if word not in self.word2index:
            self.word2index[word] = self.n_words
            self.index2word[self.n_words] = word
            self.word2count[word] = 1
            self.n_words += 1
        else:
            self.word2count[word] += 1

In [14]:
def prepareData(lang1, lang2, tokenizer1, tokenizer2):
    input_lang = Lang(lang1)
    output_lang = Lang(lang2)
    pairs = list(zip(ko_sentences_train, mt_sentences_train))
    print("Read %s sentence pairs" % len(pairs))
    for pair in pairs:
        input_lang.addSentence(pair[0], tokenizer1)
        output_lang.addSentence(pair[1], tokenizer2)

    print(f"Input Lang ({lang1}) vocabulary size: {input_lang.n_words}")
    print(f"Output Lang ({lang2}) vocabulary size: {output_lang.n_words}")

    return input_lang, output_lang, pairs

input_lang, output_lang, pairs = prepareData("ko", "en", tokenizer_ko, tokenizer_en)

Read 50000 sentence pairs
Input Lang (ko) vocabulary size: 32474
Output Lang (en) vocabulary size: 21703


In [15]:
print("### 한국어 문장 길이 통계 ###")
print(f"평균 길이: {np.mean(ko_lengths):.2f}")
print(f"최대 길이: {np.max(ko_lengths)}")
print(f"99% 백분위수 (P99): {np.percentile(ko_lengths, 99):.2f}")
print("-" * 30)

print("### 영어 문장 길이 통계 ###")
print(f"평균 길이: {np.mean(en_lengths):.2f}")
print(f"최대 길이: {np.max(en_lengths)}")
print(f"99% 백분위수 (P99): {np.percentile(en_lengths, 99):.2f}")
print("-" * 30)

# 두 언어 모두를 포함하는 통합 통계
all_lengths = ko_lengths + en_lengths
p99_combined = np.percentile(all_lengths, 99)
print(f"### 통합 길이 통계 ###")
print(f"전체 최대 길이 (100% 커버): {np.max(all_lengths)}")
print(f"전체 99% 백분위수 (P99): {p99_combined:.2f}")

# MAX_LENGTH의 새로운 기준 (P99 + SOS/EOS)
NEW_MAX_LENGTH = int(p99_combined) + 2 # +2는 SOS와 EOS 토큰을 위한 공간입니다.
print(f"--> 권장 MAX_LENGTH (99% 기준): {NEW_MAX_LENGTH}")

### 한국어 문장 길이 통계 ###
평균 길이: 11.30
최대 길이: 95
99% 백분위수 (P99): 30.00
------------------------------
### 영어 문장 길이 통계 ###
평균 길이: 11.68
최대 길이: 68
99% 백분위수 (P99): 30.00
------------------------------
### 통합 길이 통계 ###
전체 최대 길이 (100% 커버): 95
전체 99% 백분위수 (P99): 30.00
--> 권장 MAX_LENGTH (99% 기준): 32


- 최대 길이(196)**에 맞추면? -> 대부분의 문장은 짧은데, 빈 공간(PAD)만 잔뜩 채우게 되어 메모리와 속도 낭비가 심합니다.
- 99% 기준(30)**에 맞추면? -> 40 정도로만 설정해도 전체 데이터의 99%를 손실 없이 학습할 수 있습니다. (나머지 1%의 엄청 긴 문장은 잘리겠지만, 학습 효율을 위해 허용할 만한 수준이라고 판단.)

In [16]:
# (EDA 결과 반영)
# P99를 기반으로 새로운 MAX_LENGTH를 설정합니다.
MAX_LENGTH = 40 # 예시: 41
print(f"최종 MAX_LENGTH: {MAX_LENGTH}으로 설정되었습니다.")

최종 MAX_LENGTH: 40으로 설정되었습니다.


## 2. 데이터 텐서 변환 및 데이터 로더 구성 ⚙️

### (1) 문장을 텐서로 변환하는 함수 (tensorFromSentence)


- 신경망은 텍스트(단어)를 직접 처리할 수 없으며, 앞서 Lang 클래스로 부여한 정수 인덱스 시퀀스 형태로 변환해야 합니다. 또한, 모든 시퀀스는 **MAX_LENGTH**에 맞춰 패딩(Padding)되어야 합니다.

In [17]:
def tensorFromSentence(lang, sentence, tokenizer):
    indexes = [SOS_token]

    # 문장을 토큰화하고, 각 단어를 인덱스로 변환
    # 단어장에 없는 단어는 UNK_token으로 대체
    # MAX_LENGTH - 2 만큼만 사용 (SOS와 EOS 공간 확보)
    indexes += [lang.word2index.get(word, UNK_token) for word in tokenizer(sentence)[:MAX_LENGTH - 2]]

    indexes.append(EOS_token)

    # 길이가 MAX_LENGTH에 미달하는 경우, PAD_token으로 채우기 (패딩)
    while len(indexes) < MAX_LENGTH:
        indexes.append(PAD_token)

    # 최종적으로 길이를 MAX_LENGTH로 맞춘 후, PyTorch 텐서로 변환
    return torch.tensor(indexes[:MAX_LENGTH], dtype=torch.long, device=device)

### (2) 데이터 로더 생성

In [18]:
def get_dataloader(batch_size):
    # 한국어 입력 문장 전체를 텐서로 변환
    input_tensors = [tensorFromSentence(input_lang, inp, tokenizer_ko) for inp, _ in pairs]
    # 영어 목표(정답) 문장 전체를 텐서로 변환
    target_tensors = [tensorFromSentence(output_lang, tgt, tokenizer_en) for _, tgt in pairs]

    # 개별 텐서 리스트를 하나의 큰 텐서로 합침 (Stacking)
    input_tensors = torch.stack(input_tensors, dim=0)  # [num_samples, MAX_LENGTH]
    target_tensors = torch.stack(target_tensors, dim=0)  # [num_samples, MAX_LENGTH]

    # 입력 텐서와 목표 텐서를 결합하여 데이터셋 생성
    dataset = TensorDataset(input_tensors, target_tensors)

    # 데이터를 무작위로 선택하는 샘플러
    train_sampler = RandomSampler(dataset)

    # 배치 단위로 데이터를 제공하는 데이터 로더 생성
    train_dataloader = DataLoader(dataset, sampler=train_sampler, batch_size=batch_size)

    print(f"input_tensors.shape: {input_tensors.shape}, target_tensors.shape: {target_tensors.shape}")
    return train_dataloader

# 배치 크기 32로 데이터 로더 실행
train_dataloader = get_dataloader(batch_size=32)

input_tensors.shape: torch.Size([50000, 40]), target_tensors.shape: torch.Size([50000, 40])


## Seq2Seq 모델: 인코더와 디코더 구현 🏗️

In [19]:
import torch.nn.functional as F

class EncoderRNN(nn.Module):
    def __init__(self, input_size, hidden_size, dropout_p=0.1):
        super(EncoderRNN, self).__init__()
        self.hidden_size = hidden_size

        self.embedding = nn.Embedding(input_size, hidden_size)
        self.gru = nn.GRU(hidden_size, hidden_size, batch_first=True)
        self.dropout = nn.Dropout(dropout_p)

    def forward(self, input):
        # 1. 임베딩: [batch_size, MAX_LENGTH] → [batch_size, MAX_LENGTH, hidden_size]
        embedded = self.dropout(self.embedding(input))

        # 2. GRU 통과
        # output: 모든 타임스텝의 출력 (어텐션 사용 시 필요)
        # hidden: 최종 히든 상태 (컨텍스트 벡터 역할)
        output, hidden = self.gru(embedded)

        return output, hidden

In [20]:
class DecoderRNN(nn.Module):
    def __init__(self, hidden_size, output_size):
        super(DecoderRNN, self).__init__()
        self.hidden_size = hidden_size
        self.embedding = nn.Embedding(output_size, hidden_size)
        self.gru = nn.GRU(hidden_size, hidden_size, batch_first=True)
        self.out = nn.Linear(hidden_size, output_size) # output_size는 영어 단어장 크기

    def forward(self, encoder_outputs, encoder_hidden, target_tensor=None):
        batch_size = encoder_outputs.size(0)

        # 1. 초기 상태 설정
        # 초기 입력: 모든 샘플에 대해 SOS 토큰으로 시작 ([batch_size, 1])
        decoder_input = torch.empty(batch_size, 1, dtype=torch.long, device=encoder_outputs.device).fill_(SOS_token)
        decoder_hidden = encoder_hidden # 인코더의 최종 히든 상태를 초기 히든 상태로 사용
        decoder_outputs = []

        # 2. 문장 생성 루프
        for i in range(MAX_LENGTH):
            decoder_output, decoder_hidden = self.forward_step(decoder_input, decoder_hidden)
            decoder_outputs.append(decoder_output)

            # 3. 다음 입력 결정 (Teacher Forcing vs. 모델 예측)
            if target_tensor is not None:
                # Teacher forcing: 학습 시 정답(target_tensor)의 i번째 토큰을 다음 입력으로 사용
                decoder_input = target_tensor[:, i].unsqueeze(1)
            else:
                # 모델 예측: 추론 시 모델이 예측한 가장 확률 높은 토큰을 다음 입력으로 사용
                _, topi = decoder_output.topk(1)
                decoder_input = topi.squeeze(2).detach() # 예측 결과에서 인덱스 추출

        # 4. 최종 출력 정리
        # [batch_size, MAX_LENGTH, output_size]
        decoder_outputs = torch.cat(decoder_outputs, dim=1)
        decoder_outputs = F.log_softmax(decoder_outputs, dim=-1) # 손실 함수(NLLLoss)를 위한 로그 소프트맥스 적용

        return decoder_outputs, decoder_hidden, None

    def forward_step(self, input, hidden):
        # 디코더의 한 타임스텝(단어 하나 생성) 처리
        output = self.embedding(input)          # [batch_size, 1] → [batch_size, 1, hidden_size]
        output = F.relu(output)
        output, hidden = self.gru(output, hidden) # GRU를 통과시켜 다음 히든 상태 계산
        output = self.out(output)                 # [batch_size, 1, hidden_size] → [batch_size, 1, output_size] (단어장 크기만큼의 확률 분포)
        return output, hidden

## Seq2Seq 모델 학습 및 평가 함수 구성 📊

### 1) 한 에폭(Epoch) 학습 함수 (train_epoch)

In [21]:
def train_epoch(dataloader, encoder, decoder, encoder_optimizer,
                decoder_optimizer, criterion):
    encoder.train()  # 모델을 학습 모드로 설정
    decoder.train()

    total_loss = 0
    for data in dataloader:
        input_tensor, target_tensor = data

        # 1. 텐서 디바이스 이동 및 타입 변환
        input_tensor = input_tensor.long().to(device)
        target_tensor = target_tensor.long().to(device)

        # 2. 그래디언트 초기화
        encoder_optimizer.zero_grad()
        decoder_optimizer.zero_grad()

        # 3. 순전파 (Forward Pass)
        encoder_outputs, encoder_hidden = encoder(input_tensor)
        # 학습 중이므로 target_tensor를 디코더에 제공 (Teacher Forcing 활성화)
        decoder_outputs, _, _ = decoder(encoder_outputs, encoder_hidden, target_tensor)

        # 4. 손실(Loss) 계산
        # NLLLoss를 사용하기 위해 출력을 [batch_size * MAX_LENGTH, vocab_size] 형태로 변환
        loss = criterion(
            decoder_outputs.view(-1, decoder_outputs.size(-1)),
            target_tensor.view(-1)
        )

        # 5. 역전파 (Backward Pass) 및 파라미터 업데이트
        loss.backward()

        encoder_optimizer.step()
        decoder_optimizer.step()

        total_loss += loss.item()

    return total_loss / len(dataloader) # 평균 손실 반환

### 2) 전체 학습 루프 함수 (train_seq2seq)

In [22]:
def train_seq2seq(train_dataloader, encoder, decoder, n_epochs, learning_rate=0.001, print_every=100):
    print_loss_total = 0  # Reset every print_every

    # 옵티마이저 설정 (Adam이 보편적으로 좋은 성능을 보입니다.)
    encoder_optimizer = optim.Adam(encoder.parameters(), lr=learning_rate)
    decoder_optimizer = optim.Adam(decoder.parameters(), lr=learning_rate)
    # 손실 함수 설정 (디코더 출력 log_softmax와 맞추어 NLLLoss 사용)
    criterion = nn.NLLLoss()

    for epoch in range(1, n_epochs + 1):
        loss = train_epoch(train_dataloader, encoder, decoder, encoder_optimizer, decoder_optimizer, criterion)

        # 현재 에폭의 손실(Loss)을 출력
        if epoch % print_every == 0:
            print(f"Epoch {epoch}/{n_epochs}, Loss: {loss:.4f}")

### 3) 모델 평가 함수

In [23]:
def evaluate(encoder, decoder, sentence, input_lang, output_lang):
    encoder.eval() # 모델을 평가 모드로 설정
    decoder.eval()

    with torch.no_grad(): # 추론 시에는 그래디언트 계산을 비활성화 (메모리 및 속도 최적화)
        # 1. 입력 텐서 준비
        # 단일 문장이므로 배치 차원 추가 (shape: [1, MAX_LENGTH])
        input_tensor = tensorFromSentence(input_lang, sentence, tokenizer_ko).unsqueeze(0)

        # 2. 인코딩 및 디코딩 실행
        encoder_outputs, encoder_hidden = encoder(input_tensor)
        # target_tensor=None 이므로, 디코더는 자신의 예측 결과를 다음 입력으로 사용 (일반 번역 모드)
        decoder_outputs, decoder_hidden, decoder_attn = decoder(encoder_outputs, encoder_hidden)

        # 3. 예측 인덱스 추출
        _, topi = decoder_outputs.topk(1) # 가장 높은 확률을 가진 인덱스 추출
        decoded_ids = topi.squeeze() # 불필요한 차원 제거

        # 4. 인덱스를 단어로 변환
        decoded_words = []
        for idx in decoded_ids:
            item_idx = idx.item()
            if item_idx == EOS_token: # EOS 토큰이 나오면 번역 중단
                decoded_words.append('')
                break
            decoded_words.append(output_lang.index2word.get(item_idx, 'UNK')) # UNK 처리 추가

    return decoded_words, decoder_attn

In [24]:
def evaluate(encoder, decoder, sentence, input_lang, output_lang):
    encoder.eval() # 모델을 평가 모드로 설정
    decoder.eval()

    with torch.no_grad(): # 추론 시에는 그래디언트 계산을 비활성화 (메모리 및 속도 최적화)
        # 1. 입력 텐서 준비
        # 단일 문장이므로 배치 차원 추가 (shape: [1, MAX_LENGTH])
        input_tensor = tensorFromSentence(input_lang, sentence, tokenizer_ko).unsqueeze(0)

        # 2. 인코딩 및 디코딩 실행
        encoder_outputs, encoder_hidden = encoder(input_tensor)
        # target_tensor=None 이므로, 디코더는 자신의 예측 결과를 다음 입력으로 사용 (일반 번역 모드)
        decoder_outputs, decoder_hidden, decoder_attn = decoder(encoder_outputs, encoder_hidden)

        # 3. 예측 인덱스 추출
        _, topi = decoder_outputs.topk(1) # 가장 높은 확률을 가진 인덱스 추출
        decoded_ids = topi.squeeze() # 불필요한 차원 제거

        # 4. 인덱스를 단어로 변환
        decoded_words = []
        for idx in decoded_ids:
            item_idx = idx.item()
            if item_idx == EOS_token: # EOS 토큰이 나오면 번역 중단
                decoded_words.append('')
                break
            decoded_words.append(output_lang.index2word.get(item_idx, 'UNK')) # UNK 처리 추가

    return decoded_words, decoder_attn

In [25]:


# 1. 모델 인스턴스 생성
# input_lang.n_words: 한국어 단어장 크기
encoder = EncoderRNN(input_lang.n_words, HIDDEN_SIZE).to(device)
# output_lang.n_words: 영어 단어장 크기
decoder = DecoderRNN(HIDDEN_SIZE, output_lang.n_words).to(device)

print(f"✅ Baseline 모델 초기화 완료!")
print(f"- 인코더: {input_lang.n_words} (입력 크기) -> {HIDDEN_SIZE} (은닉 크기)")
print(f"- 디코더: {HIDDEN_SIZE} (은닉 크기) -> {output_lang.n_words} (출력 크기)")

✅ Baseline 모델 초기화 완료!
- 인코더: 32474 (입력 크기) -> 256 (은닉 크기)
- 디코더: 256 (은닉 크기) -> 21703 (출력 크기)


In [ ]:
# 하이퍼파라미터 설정
HIDDEN_SIZE = 256
N_EPOCHS = 10
LEARNING_RATE = 0.001

import time

start_time = time.time()
print("📚 Seq2Seq Baseline 모델 학습 시작...")

train_seq2seq(
    train_dataloader,
    encoder,
    decoder,
    N_EPOCHS,
    learning_rate=LEARNING_RATE,
    print_every=1
)

end_time = time.time()
print(f"\n✨ 학습 완료! 총 소요 시간: {end_time - start_time:.2f}초")

📚 Seq2Seq Baseline 모델 학습 시작...


In [ ]:
print("\n🔍 무작위 샘플 번역 결과 (정성적 평가)")
print("---")
# evaluateRandomly 함수는 train_data에서 샘플을 가져와 평가합니다.
evaluateRandomly(encoder, decoder, n=5)